# 中国风光水逐小时网格容量因子

本notebook使用公开ERA5再分析数据和atlite，在规则网格上生成指定年份逐小时的
风电、光伏及水电资源可用率。输出维度为`time × y × x`，供`uc_exp.ipynb`
按发电节点坐标采样。Global Wind Atlas和Global Solar Atlas适合校准长期空间
均值；逐小时变化使用ERA5。

ERA5下载需要在Copernicus Climate Data Store注册、接受ERA5许可，并配置
`~/.cdsapirc`。全国全年数据量较大，建议按月请求并保留cutout。


## 1. 集中配置

In [ ]:
from pathlib import Path

from IPython.display import display
import atlite
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

cf_year = 2024
cf_prepare_era5 = False
cf_bounds = {"x": slice(73.0, 136.0), "y": slice(18.0, 54.0)}
cf_cutout_path = Path(f"data/capacity-factors/era5_china_{cf_year}.nc")
cf_output_path = Path(f"outputs/capacity_factors_{cf_year}.nc")
cf_summary_path = Path(f"outputs/capacity_factors_{cf_year}_summary.csv")
cf_figure_path = Path(f"outputs/capacity_factors_{cf_year}_sample.png")

cf_onshore_turbine = "NREL_ReferenceTurbine_2020ATB_5.5MW"
cf_offshore_turbine = "NREL_ReferenceTurbine_2020ATB_15MW_offshore"
cf_solar_panel = "CSi"
cf_solar_orientation = "latitude_optimal"
cf_hydro_target_mean_pu = {
    "run_of_river_hydropower": 0.45,
    "reservoir_hydropower": 0.55,
    "other_hydropower": 0.50,
}
cf_sample_point = {"name": "Shandong", "x": 117.0, "y": 36.5}
cf_figure_size = (12, 6)
cf_figure_dpi = 300


## 2. 准备ERA5 cutout

atlite只下载转换所需变量：100 m风场、短波辐射、温度和地表径流。
`cf_prepare_era5=False`时只读取已经存在的cutout，不会发起网络请求。


In [ ]:
cf_cutout_path.parent.mkdir(parents=True, exist_ok=True)
cf_output_path.parent.mkdir(parents=True, exist_ok=True)

if cf_prepare_era5:
    cf_cutout = atlite.Cutout(
        cf_cutout_path,
        module="era5",
        x=cf_bounds["x"],
        y=cf_bounds["y"],
        time=slice(
            f"{cf_year}-01-01",
            f"{cf_year}-12-31 23:00",
        ),
    )
    cf_cutout.prepare(
        features=["wind", "influx", "temperature", "runoff"],
        monthly_requests=True,
        show_progress=True,
    )
elif cf_cutout_path.exists():
    cf_cutout = atlite.Cutout(cf_cutout_path)
else:
    cf_cutout = None

cf_source_summary = pd.Series({
    "atlite_version": atlite.__version__,
    "year": cf_year,
    "prepare_era5": cf_prepare_era5,
    "cutout_path": str(cf_cutout_path),
    "cutout_available": cf_cutout is not None,
    "output_path": str(cf_output_path),
}, name="value")
display(cf_source_summary)

if cf_cutout is None:
    print(
        "尚无ERA5 cutout。配置~/.cdsapirc并把"
        "cf_prepare_era5改为True后重新从本cell执行。"
    )


## 3. 生成风电和光伏逐小时容量因子

atlite根据轮毂高度风速和机组功率曲线计算风电容量因子，根据辐射、温度、组件
和朝向计算光伏容量因子。结果逐网格、逐小时位于0到1之间。


In [ ]:
cf_dataset = None
if cf_cutout is not None:
    cf_dataset = xr.Dataset({
        "onshore_wind": cf_cutout.wind(
            turbine=cf_onshore_turbine,
            capacity_factor_timeseries=True,
        ),
        "offshore_wind": cf_cutout.wind(
            turbine=cf_offshore_turbine,
            capacity_factor_timeseries=True,
        ),
        "utility_scale_solar": cf_cutout.pv(
            panel=cf_solar_panel,
            orientation=cf_solar_orientation,
            capacity_factor_timeseries=True,
        ),
    })


## 4. 构造水电资源可用率

ERA5 runoff不是电站容量因子。这里把逐网格径流转换成时间形状，再用目标年平均
可用率校准：径流式采用24小时平滑，水库型采用168小时平滑，其他水电采用72小时
平滑。该结果只能作为水文可用率代理；正式水库模型仍需HydroBASINS、入流路由、
库容、水头和调度约束。


In [ ]:
if cf_dataset is not None:
    _runoff = cf_cutout.data["runoff"].clip(min=0)
    for _technology, _window in {
        "run_of_river_hydropower": 24,
        "reservoir_hydropower": 168,
        "other_hydropower": 72,
    }.items():
        _shape = _runoff.rolling(
            time=_window, min_periods=1, center=True
        ).mean()
        _shape = _shape / _shape.mean("time").where(
            _shape.mean("time") > 0
        )
        cf_dataset[_technology] = (
            _shape * cf_hydro_target_mean_pu[_technology]
        ).clip(0, 1).fillna(0)


## 5. 校验、保存与样例图

In [ ]:
if cf_dataset is not None:
    cf_dataset = cf_dataset.astype("float32")
    cf_dataset.attrs.update({
        "source": "ERA5 via atlite",
        "year": cf_year,
        "spatial_mean_note": (
            "Wind/PV are physical conversion outputs; hydropower is "
            "a runoff-shape proxy calibrated to target annual means."
        ),
    })
    if (
        cf_dataset.to_array().isnull().any()
        or (cf_dataset.to_array() < 0).any()
        or (cf_dataset.to_array() > 1).any()
    ):
        raise ValueError("容量因子存在缺失或超出[0, 1]。")
    if len(cf_dataset.time) not in {8760, 8784}:
        raise ValueError("时间维度不是完整年份的8760或8784小时。")

    cf_summary = pd.DataFrame({
        "annual_mean_pu": cf_dataset.mean(
            ["time", "y", "x"]
        ).to_array().to_pandas(),
        "minimum_pu": cf_dataset.min(
            ["time", "y", "x"]
        ).to_array().to_pandas(),
        "maximum_pu": cf_dataset.max(
            ["time", "y", "x"]
        ).to_array().to_pandas(),
    })
    cf_dataset.to_netcdf(
        cf_output_path,
        encoding={
            variable: {"zlib": True, "complevel": 4}
            for variable in cf_dataset.data_vars
        },
    )
    cf_summary.to_csv(cf_summary_path)
    display(cf_summary)

    _sample = cf_dataset.sel(
        x=cf_sample_point["x"],
        y=cf_sample_point["y"],
        method="nearest",
    ).sel(time=slice(f"{cf_year}-07-01", f"{cf_year}-07-07"))
    _figure, _axis = plt.subplots(figsize=cf_figure_size)
    for _variable in [
        "onshore_wind", "offshore_wind",
        "utility_scale_solar", "run_of_river_hydropower",
    ]:
        _axis.plot(
            _sample.time,
            _sample[_variable],
            label=_variable.replace("_", " "),
        )
    _axis.set(
        title=(
            f"{cf_sample_point['name']} gridded resource "
            f"availability: July {cf_year}"
        ),
        ylabel="Capacity factor / availability (p.u.)",
        xlabel="Time",
        ylim=(0, 1),
    )
    _axis.grid(axis="y", alpha=0.3)
    _axis.legend(frameon=False, ncol=2)
    _figure.tight_layout()
    _figure.savefig(
        cf_figure_path,
        dpi=cf_figure_dpi,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.show()


## 6. 与UC集成

运行本notebook生成`outputs/capacity_factors_YYYY.nc`后，在`uc_exp.ipynb`
首个配置cell将`uc_resource_profile_mode`改为`"cf_file"`。UC按照每个聚合
发电组合接入bus的经纬度，从网格NetCDF就近采样对应类型的24小时曲线，并把它
作为PyPSA `Generator.p_max_pu(t)`。传统火电不读取容量因子，默认可用上限为1。

当前接口按并网节点坐标采样。若要保留GEM项目坐标，应在聚合前按
`GEM unit/phase ID`采样，再按容量加权聚合到`node_uid-generation_type`。
